In [ ]:
!pip install -U bitsandbytes transformers accelerate zeus-ml scikit-learn tqdm -q

In [1]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import torch
print("CUDA:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0))

CUDA: True | Device: Tesla T4


In [4]:
from transformers import LlavaOnevisionForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "llava-hf/llava-onevision-qwen2-7b-ov-hf"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map={"": 0},
    attn_implementation="eager",
)
model.eval()
print("Loaded. Layers:", model.config.text_config.num_hidden_layers)

processor_config.json:   0%|          | 0.00/178 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

... (6 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/765 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

Loaded. Layers: 28


In [9]:
# Cell 5b — reload processor with capped resolution (run once, before re-running the probe)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    image_grid_pinpoints=[[384, 384]],   # force single-tile, no multi-crop expansion
)

# quick check on the same sample as before
s = SAMPLES[0]
image = Image.open(os.path.join(IMG_DIR, s["image_file"])).convert("RGB")
messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": f"{s['question']} Answer the question using a single word or phrase."}]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
print("New sequence length:", len(inputs["input_ids"][0]))

New sequence length: 1342


In [11]:
# Cell 5c — test with a directly downsized image
small_image = image.resize((336, 336))
inputs_small = processor(text=[text], images=[small_image], return_tensors="pt").to(model.device)
print("Sequence length with resized image:", len(inputs_small["input_ids"][0]))

Sequence length with resized image: 1510


In [15]:
# Cell 5d — sync model config to match the processor's override
model.config.image_grid_pinpoints = [[384, 384]]
if hasattr(model.config, "vision_config"):
    model.config.vision_config.image_grid_pinpoints = [[384, 384]]

# re-verify sequence length is consistent now
s = SAMPLES[0]
image = Image.open(os.path.join(IMG_DIR, s["image_file"])).convert("RGB")
messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": f"{s['question']} Answer the question using a single word or phrase."}]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
print("Sequence length:", len(inputs["input_ids"][0]))

with torch.no_grad():
    _ = model(**inputs, use_cache=False)   # no output_attentions yet - just confirm no split error
print("Forward pass succeeded, no split mismatch")

Sequence length: 1342
Forward pass succeeded, no split mismatch


In [5]:
# Cell 6 — verify image tokens actually expand correctly
import os, json
from PIL import Image

DATA_DIR = "/kaggle/input/notebooks/minhajurrahman223050/ecovlm-data-prep/vqav2_subset"
IMG_DIR = os.path.join(DATA_DIR, "images")
with open(os.path.join(DATA_DIR, "metadata.json")) as f:
    SAMPLES = json.load(f)

s = SAMPLES[0]
image = Image.open(os.path.join(IMG_DIR, s["image_file"])).convert("RGB")
messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": f"{s['question']} Answer the question using a single word or phrase."}]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

ids = inputs["input_ids"][0]
print("Total sequence length:", len(ids))
print("model.config attributes with 'image' or 'token':")
for attr in dir(model.config):
    if 'image' in attr.lower() and not attr.startswith('_'):
        print(f"  {attr} = {getattr(model.config, attr, None)}")

img_tok_id = getattr(model.config, "image_token_id", getattr(model.config, "image_token_index", None))
print()
print("Using image token id:", img_tok_id)
print("Image placeholder count:", (ids == img_tok_id).sum().item() if img_tok_id is not None else "UNKNOWN - check attrs above")

Total sequence length: 1753
model.config attributes with 'image' or 'token':
  image_grid_pinpoints = [[384, 384], [384, 768], [384, 1152], [384, 1536], [384, 1920], [384, 2304], [768, 384], [768, 768], [768, 1152], [768, 1536], [768, 1920], [768, 2304], [1152, 384], [1152, 768], [1152, 1152], [1152, 1536], [1152, 1920], [1152, 2304], [1536, 384], [1536, 768], [1536, 1152], [1536, 1536], [1536, 1920], [1536, 2304], [1920, 384], [1920, 768], [1920, 1152], [1920, 1536], [1920, 1920], [1920, 2304], [2304, 384], [2304, 768], [2304, 1152], [2304, 1536], [2304, 1920], [2304, 2304]]
  image_token_index = 151646
  use_image_newline_parameter = True

Using image token id: 151646
Image placeholder count: 1728


In [6]:
# Cell 7 — profiling function (mirrors the Qwen version exactly, same 28-layer windows)
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

EARLY = list(range(2, 9))
LATE  = list(range(15, 26))
IMG_TOKEN_ID = model.config.image_token_index

def llava_attention_profile(sample):
    image = Image.open(os.path.join(IMG_DIR, sample["image_file"])).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": f"{sample['question']} Answer the question using a single word or phrase."}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

    ids = inputs["input_ids"][0]
    vis_mask = (ids == IMG_TOKEN_ID)
    vis_idx = torch.nonzero(vis_mask).squeeze(-1)
    if vis_idx.numel() == 0:
        return None
    vis_start, vis_end = vis_idx[0].item(), vis_idx[-1].item() + 1
    text_start = vis_end

    with torch.no_grad():
        out = model(**inputs, output_attentions=True, use_cache=False)

    early_stack, late_stack = [], []
    for i in EARLY:
        a = out.attentions[i][0].float().mean(dim=0)
        early_stack.append(a[text_start:, vis_start:vis_end].mean(dim=0).cpu().numpy())
    for i in LATE:
        a = out.attentions[i][0].float().mean(dim=0)
        late_stack.append(a[text_start:, vis_start:vis_end].mean(dim=0).cpu().numpy())

    del out
    torch.cuda.empty_cache()
    early_stack = np.stack(early_stack)
    late_stack = np.stack(late_stack)
    return dict(early_mean=early_stack.mean(axis=0), early_var=early_stack.var(axis=0),
                late_mean=late_stack.mean(axis=0))

In [10]:
# Cell 8 — run the probe
N_PROBE = 100
llava_records = []
for s in tqdm(SAMPLES[:N_PROBE]):
    try:
        r = llava_attention_profile(s)
        if r is not None:
            llava_records.append(r)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print("OOM on", s["sample_idx"])
        continue
print("Collected:", len(llava_records))

  0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:944: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


RuntimeError: split_with_sizes expects split_sizes to sum exactly to 2 (input tensor's size at dimension 0), but got split_sizes=[3]

In [8]:
# Cell 9 — verdict
import statistics as st
aucs_mean, aucs_var = [], []
for r in llava_records:
    em, ev, lm = r["early_mean"], r["early_var"], r["late_mean"]
    pool = em <= np.percentile(em, 50)
    if pool.sum() < 20: continue
    label = (lm[pool] >= np.percentile(lm, 75)).astype(int)
    if label.sum() == 0 or label.sum() == len(label): continue
    aucs_mean.append(roc_auc_score(label, em[pool]))
    aucs_var.append(roc_auc_score(label, ev[pool]))

print(f"Samples analysed: {len(aucs_mean)}")
print(f"AUC, early mean alone : {st.mean(aucs_mean):.4f}")
print(f"AUC, variance alone   : {st.mean(aucs_var):.4f}")
print(f"DELTA: {st.mean(aucs_var) - st.mean(aucs_mean):+.4f}")

Samples analysed: 10
AUC, early mean alone : 0.6465
AUC, variance alone   : 0.6503
DELTA: +0.0038


In [12]:
# Cell 10 — hook-based selective attention capture (fixes the OOM properly)
import torch

captured = {}

def make_hook(layer_idx, wanted_layers):
    def hook(module, inputs, output):
        # output is (attn_output, attn_weights, past_kv) for eager attention
        if layer_idx in wanted_layers and len(output) > 1 and output[1] is not None:
            w = output[1][0].float().mean(dim=0).cpu().numpy()   # avg heads -> (q,k)
            captured[layer_idx] = w
        # replace the heavy GPU tensor with a lightweight placeholder either way,
        # so unwanted layers never let their full tensor persist in the accumulated tuple
        new_output = (output[0], None) + tuple(output[2:])
        return new_output
    return hook

def find_attn_modules(model):
    # LLaVA-OneVision's LLM backbone is Qwen2 - locate its decoder layers
    text_model = model.language_model if hasattr(model, "language_model") else model.model.language_model
    return text_model.layers

layers_module = find_attn_modules(model)
print("Total decoder layers found:", len(layers_module))

WANTED = set(EARLY) | set(LATE)
hook_handles = []
for i, layer in enumerate(layers_module):
    h = layer.self_attn.register_forward_hook(make_hook(i, WANTED))
    hook_handles.append(h)

print("Hooks registered on", len(hook_handles), "layers")

Total decoder layers found: 28
Hooks registered on 28 layers


In [13]:
# Cell 11 — profiling function using the hook-captured weights instead of out.attentions
def llava_attention_profile_v2(sample):
    image = Image.open(os.path.join(IMG_DIR, sample["image_file"])).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": f"{sample['question']} Answer the question using a single word or phrase."}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

    ids = inputs["input_ids"][0]
    vis_mask = (ids == IMG_TOKEN_ID)
    vis_idx = torch.nonzero(vis_mask).squeeze(-1)
    if vis_idx.numel() == 0:
        return None
    vis_start, vis_end = vis_idx[0].item(), vis_idx[-1].item() + 1
    text_start = vis_end

    captured.clear()
    with torch.no_grad():
        model(**inputs, output_attentions=True, use_cache=False)
    torch.cuda.empty_cache()

    if not all(i in captured for i in EARLY + LATE):
        return None   # some layer failed to capture, skip rather than silently mis-score

    early_stack = np.stack([captured[i][text_start:, vis_start:vis_end].mean(axis=0) for i in EARLY])
    late_stack  = np.stack([captured[i][text_start:, vis_start:vis_end].mean(axis=0) for i in LATE])

    return dict(early_mean=early_stack.mean(axis=0), early_var=early_stack.var(axis=0),
                late_mean=late_stack.mean(axis=0))

In [16]:
# Cell 12 — re-run the probe with the fixed capture mechanism
N_PROBE = 100
llava_records = []
for s in tqdm(SAMPLES[:N_PROBE]):
    try:
        r = llava_attention_profile_v2(s)
        if r is not None:
            llava_records.append(r)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print("OOM on", s["sample_idx"])
        continue
print("Collected:", len(llava_records))

  0%|          | 0/100 [00:00<?, ?it/s]

Collected: 100


In [17]:
# Cell 13 — verdict
import statistics as st
aucs_mean, aucs_var = [], []
for r in llava_records:
    em, ev, lm = r["early_mean"], r["early_var"], r["late_mean"]
    pool = em <= np.percentile(em, 50)
    if pool.sum() < 20: continue
    label = (lm[pool] >= np.percentile(lm, 75)).astype(int)
    if label.sum() == 0 or label.sum() == len(label): continue
    aucs_mean.append(roc_auc_score(label, em[pool]))
    aucs_var.append(roc_auc_score(label, ev[pool]))

print(f"Samples analysed: {len(aucs_mean)}")
print(f"AUC, early mean alone : {st.mean(aucs_mean):.4f}")
print(f"AUC, variance alone   : {st.mean(aucs_var):.4f}")
print(f"DELTA: {st.mean(aucs_var) - st.mean(aucs_mean):+.4f}")

Samples analysed: 100
AUC, early mean alone : 0.6490
AUC, variance alone   : 0.6461
DELTA: -0.0029
